# SWE015 – Chat moderation: full training pipeline

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.

Total time ≈ 30–40 minutes.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/Humeyyraa/swe015-chat-moderation.git
%cd swe015-chat-moderation
!pip install -q -r requirements.txt

## 1. Data (OffensEval-TR 2020)

In [ ]:
!python scripts/prepare_data.py

## 2. Model 1 – classical baselines (lexicon + edit distance, tf-idf + LogReg)

In [ ]:
!python scripts/train_baselines.py

## 3. Model 2 – own word2vec + feedforward network (SGD)

In [ ]:
!python scripts/train_w2v_ffn.py

## 4. Model 3 – BERTurk fine-tuning (GPU, ~10–15 min)

In [ ]:
!python scripts/train_bert.py

## 5. Model 4 – ONNX export + dynamic INT8 quantization

In [ ]:
!python scripts/quantize_bert.py

## 6. Benchmark on the test set (CPU only)
Latency is measured on the CPU of this Colab machine; the hardware is printed and saved in `results/results.json`.

In [ ]:
!python scripts/benchmark.py

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/results.md').read()))
display(Image('results/results_chart.png'))

## 7. Upload the weights to the Hugging Face Hub
Paste your **write** token when asked.

In [ ]:
import os
os.environ['HF_REPO_ID'] = 'humeyrakoctemiz/swe015-chat-moderation'
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python scripts/upload_models.py

## 8. Download the results folder (commit it to GitHub)

In [ ]:
!zip -qr results.zip results models/*/meta.json
from google.colab import files
files.download('results.zip')

## 9. (Optional) Try the app from Colab
Opens a public `gradio.live` link. Stop the cell to end it.

In [ ]:
!python app.py --share